# Capstone · Churn-voorspelling
### Classificatie-training · Antwoordmodel-versie

Dit is de afsluitende opdracht. Je past nu alles toe op een **realistische, ongebalanceerde** dataset:
een synthetisch churn-probleem dat lijkt op wat je in de praktijk tegenkomt (een minderheid van klanten
vertrekt, de meerderheid blijft).

**De stappen:**

1. Data verkennen en de **klassebalans** checken (sterk onbalans!)
2. Een eerlijke (gestratificeerde) train/test-split
3. Een **baseline** plus **twee echte modellen** bouwen
4. Evalueren met **meerdere metrieken** — niet alleen accuracy
5. De **drempel** kiezen op basis van het probleem
6. **Feature importance** bekijken: welke klantkenmerken voorspellen churn?

> **Werkvorm:** in groepjes · rest van de middag · sluit af met een presentatie van 5 minuten.


## 0. Voorbereiding & data genereren

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)

sns.set_theme(style="whitegrid")
print("Bibliotheken geladen.")

In [ ]:
# Synthetisch churn-dataset (geen download nodig)
X_raw, y_raw = make_classification(
    n_samples=2000, n_features=8, n_informative=5, n_redundant=2,
    weights=[0.85, 0.15],   # 85% blijft, 15% churnt
    flip_y=0.03, class_sep=1.0, random_state=42
)
feat_names = ["maanden_klant", "aantal_producten", "klachten_3m", "login_freq",
              "laatste_betaling_dgn", "klantsegment_score", "contract_lengte", "support_calls"]
X = pd.DataFrame(X_raw, columns=feat_names)
y = pd.Series(y_raw, name="churn")

print("Aantal klanten:", len(X))
print("Features:", feat_names)

## 1. Data verkennen en klassebalans

### TODO 1 — Bekijk de klassebalans
Tel hoeveel klanten in elke klasse zitten (0 = blijft, 1 = churn).

In [ ]:
# ANTWOORD
balans = y.value_counts()
print(balans)
print(f"\nChurn-percentage: {balans[1] / balans.sum() * 100:.1f}%")

balans.plot(kind="bar", color=["#4473C5","#D97733"])
plt.xticks([0,1], ["blijft (0)","churn (1)"], rotation=0)
plt.title("Klassebalans"); plt.ylabel("Aantal klanten"); plt.show()

> **Let op:** met ~85% “blijft”, kun je een accuracy van 85% halen door **altijd “blijft”
> te voorspellen**. Dat is een waardeloos model — daarom moeten we anders evalueren.

## 2. Train/test-split (gestratificeerd)

### TODO 2 — Maak een gestratificeerde split (80/20)

In [ ]:
# ANTWOORD
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Churn in train: {y_train.mean()*100:.1f}%  |  Churn in test: {y_test.mean()*100:.1f}%")

## 3. Baseline + twee echte modellen
Bouw een **baseline** (meerderheidsklasse), een **logistisch model** en een **random forest**.

### TODO 3 — Vul de dictionary aan met de drie modellen

In [ ]:
# ANTWOORD
modellen = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistisch": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

> Let op: voor de logistische en forest geven we `class_weight="balanced"` mee. Dat compenseert
> voor de onbalans — het model krijgt een straf voor het missen van de minderheidsklasse.

## 4. Evalueren met meerdere metrieken
We trainen elk model en berekenen accuracy, precision, recall, F1 en AUC.

In [ ]:
# Evaluatie-lus (al ingevuld)
resultaten = []
for naam, model in modellen.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    # Voor AUC hebben we kansen nodig; baseline heeft alleen één klasse
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, proba)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan
    resultaten.append({
        "model": naam,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "AUC": auc,
    })

resultaten = pd.DataFrame(resultaten).set_index("model")
print(resultaten.round(3))

> **Let goed op:** de baseline haalt een accuracy van ~85% (alleen omdat hij altijd “blijft” zegt),
> maar zijn recall is 0 — hij vangt **geen enkele churner**. Dit is waarom je nooit alleen op accuracy
> moet vertrouwen bij onbalans.

## 5. Confusion matrix van het beste model

### TODO 4 — Maak een confusion matrix voor het random forest
De code voor de heatmap staat klaar; vul alleen de berekening van de matrix in.

In [ ]:
# ANTWOORD
forest = modellen["Random forest"]
pred_forest = forest.predict(X_test)
cm = confusion_matrix(y_test, pred_forest)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["blijft","churn"], yticklabels=["blijft","churn"], cbar=False,
            annot_kws={"size":13,"weight":"bold"})
plt.xlabel("Voorspeld"); plt.ylabel("Werkelijk")
plt.title("Confusion matrix - Random forest")
plt.show()

## 6. Drempelkeuze: precision-recall-trade-off

In [ ]:
# Precision en recall bij verschillende drempels (al ingevuld)
proba = forest.predict_proba(X_test)[:, 1]
drempels = np.linspace(0.1, 0.9, 17)
prec_l, rec_l = [], []
for d in drempels:
    pred_d = (proba >= d).astype(int)
    prec_l.append(precision_score(y_test, pred_d, zero_division=0))
    rec_l.append(recall_score(y_test, pred_d, zero_division=0))

plt.figure(figsize=(7, 4))
plt.plot(drempels, prec_l, marker="o", color="#4473C5", label="Precision")
plt.plot(drempels, rec_l, marker="s", color="#D97733", label="Recall")
plt.xlabel("Drempel"); plt.ylabel("Score")
plt.title("Precision vs. Recall over de drempel - churn")
plt.legend(); plt.show()

### TODO 5 — Kies een drempel
Op basis van de plot: welke drempel zou jij kiezen voor churn-voorspelling?
Bedenk dat een gemiste churner duurder is dan een onnodige outreach.

In [ ]:
# ANTWOORD
gekozen_drempel = 0.3  # voorbeeld - lager dan default 0.5 om meer churners te vangen
pred_gekozen = (proba >= gekozen_drempel).astype(int)
print(f"Bij drempel {gekozen_drempel}:")
print(f"  Precision: {precision_score(y_test, pred_gekozen, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_test, pred_gekozen, zero_division=0):.3f}")

## 7. Feature importance: welke klantkenmerken voorspellen churn?

### TODO 6 — Maak een bar plot van de feature importances

In [ ]:
# ANTWOORD
importance = pd.Series(forest.feature_importances_, index=feat_names).sort_values(ascending=False)
print(importance.round(3))

plt.figure(figsize=(7, 4.5))
sns.barplot(x=importance.values, y=importance.index, color="#4473C5", edgecolor="#203864")
plt.xlabel("Belang"); plt.title("Welke klantkenmerken voorspellen churn?")
plt.show()

## 8. Presenteer (5 minuten)
Bereid een korte presentatie voor met:

- De klassebalans en waarom accuracy hier misleidend is
- Welke modellen je hebt vergeleken en de cijfers (precision, recall, AUC)
- Welke **drempel** je hebt gekozen en waarom
- De **top 3 features** die churn voorspellen
- Eén ding dat je verraste

> Beoordeel elkaar vooral op de **aanpak en redenering**, niet alleen op de scores.

---
### Toelichting voor de trainer
- **Klassebalans**: ~85% blijft, ~15% churn (realistisch).
- **Baseline accuracy** ~0.85 maar **recall = 0** (vangt geen enkele churner) — kernpunt.
- **Random forest met class_weight=balanced**: accuracy ~0.93, **recall ~0.66**, precision ~0.90, AUC ~0.95.
  Veel beter dan baseline op de metrieken die ertoe doen.
- **Logistisch met balanced** doet minder goed: tussen baseline en random forest in. Goed gesprekspunt:
  niet-lineaire patronen vragen vaak om tree-based modellen.
- **Drempel**: bij 0.3 stijgt recall sterk (meer churners gevangen) ten koste van precision (meer onterechte
  outreach). Bespreek de business case: kost van outreach vs. kost van gemiste klant.